# Prediction & Credit Scoring

This notebook demonstrates the end-to-end prediction workflow:

1. **Load** the trained model and fitted pipeline from MLflow/registry
2. **Score** customers with risk probability
3. **Convert** probabilities to credit scores (300-850)
4. **Recommend** loan amounts and durations

## 1. Setup

In [1]:
import sys
from pathlib import Path

proj_root = Path.cwd().resolve().parent
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.predict import (
    load_model,
    load_pipeline,
    predict_risk,
    predict_single,
    probability_to_credit_score,
    recommend_loan,
)

sns.set_theme(style="whitegrid")

## 2. Load Model and Pipeline

In [2]:
from src.train import train_and_track
train_and_track('../data/processed/processed_data.csv')

2026-07-15 15:36:30,266 [INFO] Embedded WoE/IV utilities into data_processing module
2026-07-15 15:36:30,271 [INFO] Loading processed data from: ../data/processed/processed_data.csv
2026-07-15 15:36:31,939 [INFO] Dropped non-feature columns: ['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'CurrencyCode', 'TransactionStartTime', 'FirstTransaction', 'LastTransaction']
2026-07-15 15:36:32,173 [INFO] Features: 62 columns


Split completed:
  Train: 66,963 records(70.0%)
  Val:  9,566 records(10.0%)
  Test:  19,133 records(20.0%)
  Target distribution (Train): {0.0: 0.884, 1.0: 0.116}


2026-07-15 15:36:33,831 [INFO] Training LogisticRegression...
c:\Users\teMelkishi\Desktop\ai\credit-risk-model\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/07/15 15:36:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026-07-15 15:37:17,359 [INFO] LogisticRegression - Test ROC-AUC: 1.0000
2026-07-15 15:37:17,397 [INFO] Training RandomForest...
2026/07/15 15:37:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Pl

{'LogisticRegression': {'model': LogisticRegression(max_iter=1000, random_state=42),
  'test_metrics': {'accuracy': 0.998484294151466,
   'precision': 0.9936823104693141,
   'recall': 0.993234100135318,
   'f1': 0.9934581547484773,
   'roc_auc': 0.9999691489471766},
  'run_id': '76d4c893dca247d6b0775fe7f5d77fdf'},
 'RandomForest': {'model': RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
  'test_metrics': {'accuracy': 0.9998432028432551,
   'precision': 1.0,
   'recall': 0.9986468200270636,
   'f1': 0.999322951929587,
   'roc_auc': 1.0},
  'run_id': '0467be6d364e4b38a1dffc47d59cb7e1'},
 'GradientBoosting': {'model': GradientBoostingClassifier(n_estimators=200, random_state=42),
  'test_metrics': {'accuracy': 1.0,
   'precision': 1.0,
   'recall': 1.0,
   'f1': 1.0,
   'roc_auc': 1.0},
  'run_id': 'f649a443a40f44b99c894e084f33306e'},
 'XGBoost': {'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_

In [3]:
model = load_model()
pipeline = load_pipeline()
print("Model and pipeline loaded successfully.")

2026-07-15 15:38:58,805 [INFO] Trying Model Registry: models:/credit-risk-best-model/1
2026-07-15 15:38:59,621 [INFO] Model loaded from Model Registry.
2026-07-15 15:38:59,624 [INFO] Loading pipeline from: C:\Users\teMelkishi\Desktop\ai\credit-risk-model\models\pipeline.joblib


Model and pipeline loaded successfully.


## 3. Load Processed Data

In [4]:
data_path = proj_root / "data" / "processed" / "processed_data.csv"
df = pd.read_csv(data_path)

target_col = "is_high_risk"
df = df.drop(columns=[c for c in ['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'CurrencyCode', 'TransactionStartTime', 'FirstTransaction', 'LastTransaction'] if c in df.columns])
feature_cols = [c for c in df.columns if c != target_col]

print(f"Dataset shape: {df.shape}")
print(f"Features: {len(feature_cols)}")
print(f"Target distribution:")
print(df[target_col].value_counts(normalize=True).round(3))


Dataset shape: (95662, 63)
Features: 62
Target distribution:
is_high_risk
0.0    0.884
1.0    0.116
Name: proportion, dtype: float64


## 4. Batch Predictions

Score all customers in the processed dataset.

In [5]:
X = df[feature_cols]
y_true = df[target_col]

risk_probs = predict_risk(model, X)
credit_scores = [probability_to_credit_score(p) for p in risk_probs]

results = df[["CustomerId"]].copy()
results["true_label"] = y_true.values
results["risk_probability"] = risk_probs
results["credit_score"] = credit_scores

results.head(10)

KeyError: "None of [Index(['CustomerId'], dtype='object')] are in the [columns]"

## 5. Prediction Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(risk_probs, bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Risk Probability Distribution")
axes[0].set_xlabel("P(Default)")
axes[0].set_ylabel("Count")
axes[0].axvline(x=0.5, color="red", linestyle="--", alpha=0.7, label="Threshold = 0.5")
axes[0].legend()

axes[1].hist(credit_scores, bins=50, color="darkorange", edgecolor="white")
axes[1].set_title("Credit Score Distribution")
axes[1].set_xlabel("Score (300-850)")
axes[1].set_ylabel("Count")
axes[1].axvline(x=500, color="red", linestyle="--", alpha=0.7, label="Medium risk threshold")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Risk vs Actual Label

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
results.boxplot(column="risk_probability", by="true_label", ax=ax)
ax.set_title("Risk Probability by True Label")
ax.set_xlabel("is_high_risk")
ax.set_ylabel("Predicted Risk Probability")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 7. Single Customer Walkthrough

Score a single customer and generate a full credit report.

In [ ]:
sample_idx = 0
sample_features = X.iloc[[sample_idx]]
sample_amounts = df["TotalTransactionAmount"].iloc[[sample_idx]]

prediction = predict_single(model, sample_features, sample_amounts)

print("=" * 50)
print("CREDIT RISK REPORT")
print("=" * 50)
print(f"Customer ID:              {df['CustomerId'].iloc[sample_idx]}")
print(f"Risk Probability:         {prediction['risk_probability']:.4f}")
print(f"Credit Score:             {prediction['credit_score']}")
print(f"Recommended Loan Amount:  {prediction['recommended_loan_amount']}")
print(f"Loan Duration:            {prediction['recommended_loan_duration_months']} months")
print("=" * 50)

## 8. Batch Loan Recommendations

Generate loan recommendations for a group of customers.

In [ ]:
n_customers = 20
sample_batch = X.head(n_customers)
sample_amounts_batch = df["TotalTransactionAmount"].head(n_customers)

batch_results = []
for i in range(n_customers):
    res = predict_single(
        model,
        sample_batch.iloc[[i]],
        sample_amounts_batch.iloc[[i]],
    )
    res["customer_id"] = df["CustomerId"].iloc[i]
    batch_results.append(res)

batch_df = pd.DataFrame(batch_results)
batch_df = batch_df[["customer_id", "risk_probability", "credit_score",
                      "recommended_loan_amount", "recommended_loan_duration_months"]]
batch_df

## 9. Risk Category Breakdown

In [ ]:
def risk_category(score):
    if score >= 700:
        return "Low"
    elif score >= 500:
        return "Medium"
    else:
        return "High"

results["risk_category"] = results["credit_score"].apply(risk_category)

category_counts = results["risk_category"].value_counts()
print("Risk Category Distribution:")
print(category_counts)

fig, ax = plt.subplots(figsize=(6, 4))
colors = {"Low": "green", "Medium": "orange", "High": "red"}
category_counts.plot(
    kind="bar", ax=ax,
    color=[colors[c] for c in category_counts.index],
    edgecolor="white",
)
ax.set_title("Risk Category Distribution")
ax.set_ylabel("Count")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 10. Score-to-Probability Mapping

Visualize the relationship between credit score and risk probability.

In [ ]:
probs = np.linspace(0.01, 0.99, 200)
scores = [probability_to_credit_score(p) for p in probs]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(probs, scores, color="steelblue", linewidth=2)
ax.set_xlabel("Risk Probability P(Default)")
ax.set_ylabel("Credit Score")
ax.set_title("Risk Probability to Credit Score Mapping")
ax.axhline(y=700, color="green", linestyle="--", alpha=0.5, label="Low risk (>=700)")
ax.axhline(y=500, color="orange", linestyle="--", alpha=0.5, label="Medium risk (>=500)")
ax.axhline(y=300, color="red", linestyle="--", alpha=0.5, label="High risk (<500)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

- The model assigns risk probabilities based on transaction behavior features.
- Credit scores are derived from risk probabilities using a log-odds scaling (300-850).
- Loan recommendations are based on credit score tiers and average transaction amounts.
- The `predict_single` function provides a complete credit decision for individual customers.
- The same logic powers the `/predict` endpoint in the FastAPI service.